# Thames Ecosystem & Climate Risk
## Connecting River, Climate, and Ecological Data

This notebook ties together our key datasets to tell a coherent story:

1. **The River** — Thames geography from OS Open Rivers (193k links)
2. **Monitoring Stations** — EA Hydrology stations and water quality readings
3. **Climate Signal** — Met Office pseudo-observation air temperature & precipitation (116 sites, hourly, 2011–2026)
4. **Ecological Impact** — Fish thermal tolerance thresholds and population simulations

**The narrative:** Climate warming → rising river water temperatures → thermal stress on Thames fish populations.

We join spatial, temporal, and biological data to quantify this risk.


---
## Part 1: The River — Geography & Monitoring Network

The Thames river network from OS Open Rivers, plus the EA hydrology monitoring stations that observe it.


In [ ]:
-- Thames river network summary (from OS Open Rivers)
SELECT
    COUNT(*)                          AS total_links,
    ROUND(SUM(LENGTH) / 1000, 1)     AS total_length_km,
    COUNT(DISTINCT WATERCOURSE_NAME)  AS distinct_named_sections
FROM MET_SCRATCH.THAMES.RIVER_THAMES;


In [ ]:
-- EA monitoring stations on/near the Thames
SELECT
    s.LABEL,
    s.RIVER_NAME,
    s.LAT,
    s.LONG,
    COUNT(m.ID) AS num_measures
FROM MET_SCRATCH.THAMES.STATIONS s
LEFT JOIN MET_SCRATCH.THAMES.MEASURES m ON m.STATION_ID = s.ID
WHERE s.RIVER_NAME ILIKE '%thames%'
GROUP BY 1, 2, 3, 4
ORDER BY s.LABEL;


In [ ]:
-- What parameters do our Thames stations measure?
SELECT
    m.PARAMETER,
    m.PERIOD_NAME,
    m.VALUE_TYPE,
    m.UNIT_NAME,
    COUNT(*) AS num_measures,
    COUNT(DISTINCT m.STATION_ID) AS num_stations
FROM MET_SCRATCH.THAMES.MEASURES m
JOIN MET_SCRATCH.THAMES.STATIONS s ON s.ID = m.STATION_ID
WHERE s.RIVER_NAME ILIKE '%thames%'
GROUP BY 1, 2, 3, 4
ORDER BY num_stations DESC, m.PARAMETER;


---
## Part 2: Water Temperature — What the River Actually Measures

Direct water temperature observations from EA monitoring stations on the Thames.


In [ ]:
-- Daily mean water temperature from EA stations (last 30 days of data)
SELECT
    DATE_TRUNC('day', r.DATE_TIME) AS day,
    ROUND(AVG(r.VALUE), 2) AS mean_water_temp_c,
    ROUND(MAX(r.VALUE), 2) AS max_water_temp_c,
    COUNT(DISTINCT r.STATION_ID) AS reporting_stations
FROM MET_SCRATCH.THAMES.READINGS r
JOIN MET_SCRATCH.THAMES.STATIONS s ON s.ID = r.STATION_ID
WHERE r.PARAMETER = 'TEMPERATURE'
  AND s.RIVER_NAME ILIKE '%thames%'
GROUP BY 1
ORDER BY day DESC
LIMIT 30;


In [ ]:
-- Per-station water temperature time series (most recent year, daily)
SELECT
    s.LABEL AS station,
    DATE_TRUNC('day', r.DATE_TIME) AS day,
    ROUND(AVG(r.VALUE), 2) AS avg_water_temp_c
FROM MET_SCRATCH.THAMES.READINGS r
JOIN MET_SCRATCH.THAMES.STATIONS s ON s.ID = r.STATION_ID
WHERE r.PARAMETER = 'TEMPERATURE'
  AND r.PERIOD_NAME = '15min'
  AND s.RIVER_NAME ILIKE '%thames%'
GROUP BY 1, 2
ORDER BY station, day;


---
## Part 3: Air Temperature & Precipitation — Met Office Climate Signal

116 Met Office pseudo-observation sites along the Thames corridor, hourly data from 2011–2026.
Air temperature is a proxy/driver of water temperature.


In [ ]:
-- Coverage summary
SELECT
    COUNT(*) AS total_observations,
    COUNT(DISTINCT SITE_ID) AS sites,
    MIN(VALIDITY_TIME) AS first_obs,
    MAX(VALIDITY_TIME) AS last_obs,
    ROUND(AVG(SCREEN_TEMPERATURE_C), 1) AS mean_air_temp_c,
    ROUND(MAX(SCREEN_TEMPERATURE_C), 1) AS max_air_temp_c
FROM SNOWFLAKE_INTELLIGENCE.PUBLIC.THAMES_TEMPERATURE_TIMESERIES;


In [ ]:
-- Annual warming trend: is Thames corridor air temperature rising?
SELECT
    YEAR(VALIDITY_TIME) AS yr,
    ROUND(AVG(SCREEN_TEMPERATURE_C), 2) AS mean_annual_air_temp_c,
    ROUND(MAX(SCREEN_TEMPERATURE_C), 1) AS max_air_temp_c,
    COUNT(*) AS hourly_obs
FROM SNOWFLAKE_INTELLIGENCE.PUBLIC.THAMES_TEMPERATURE_TIMESERIES
GROUP BY 1
ORDER BY 1;


In [ ]:
-- Summer heat exceedance: hours above key fish stress thresholds per year
-- 26°C = upper optimum for most Thames species
-- 30°C = onset of thermal stress
-- 33°C+ = approaching lethal range
SELECT
    YEAR(VALIDITY_TIME) AS yr,
    COUNT_IF(SCREEN_TEMPERATURE_C >= 26) AS hours_above_26c,
    COUNT_IF(SCREEN_TEMPERATURE_C >= 30) AS hours_above_30c,
    COUNT_IF(SCREEN_TEMPERATURE_C >= 33) AS hours_above_33c
FROM SNOWFLAKE_INTELLIGENCE.PUBLIC.THAMES_TEMPERATURE_TIMESERIES
WHERE MONTH(VALIDITY_TIME) BETWEEN 5 AND 9
GROUP BY 1
ORDER BY 1;


---
## Part 4: Fish Thermal Risk — Connecting Climate to Ecology

The top 10 Thames fish species have well-characterised thermal tolerance bands.
Cross-joining those thresholds against 15 years of observed air temperatures quantifies
how many site-hours of stress and lethal exposure each species has already experienced.


In [ ]:
-- Top 10 Thames fish species and their thermal tolerance bands
SELECT
    SPECIES_RANK,
    COMMON_NAME,
    THAMES_REACH,
    OPTIMUM_TEMP_MIN_C,
    OPTIMUM_TEMP_MAX_C,
    STRESS_TEMP_MAX_C,
    LETHAL_TEMP_MAX_C
FROM FISHDB.FISH_DEATH_RATES.THAMES_10
ORDER BY LETHAL_TEMP_MAX_C;


In [ ]:
-- Heat-stress exposure per species (air temp as upper-bound proxy for water temp)
SELECT
    f.COMMON_NAME,
    f.STRESS_TEMP_MAX_C,
    f.LETHAL_TEMP_MAX_C,
    COUNT_IF(t.SCREEN_TEMPERATURE_C >= f.STRESS_TEMP_MAX_C
             AND t.SCREEN_TEMPERATURE_C < f.LETHAL_TEMP_MAX_C) AS stress_site_hours,
    COUNT_IF(t.SCREEN_TEMPERATURE_C >= f.LETHAL_TEMP_MAX_C)     AS lethal_site_hours,
    ROUND(MAX(t.SCREEN_TEMPERATURE_C), 1)                       AS max_observed_air_c
FROM FISHDB.FISH_DEATH_RATES.THAMES_10 f
CROSS JOIN SNOWFLAKE_INTELLIGENCE.PUBLIC.THAMES_TEMPERATURE_TIMESERIES t
GROUP BY 1, 2, 3
ORDER BY lethal_site_hours DESC;


In [ ]:
-- Population simulation: what happens under climate warming scenarios?
SELECT
    SPECIES,
    SCENARIO,
    INITIAL_POP,
    FINAL_POP,
    TOTAL_THERMAL_DEATHS,
    ROUND((FINAL_POP - INITIAL_POP)::FLOAT / INITIAL_POP * 100, 1) AS pct_pop_change
FROM SNOWFLAKE_INTELLIGENCE.PUBLIC.THAMES_FISH_POPULATION_SUMMARY
ORDER BY SPECIES, SCENARIO;


---
## Part 5: Spatial Join — Linking Met Sites to EA Stations

Each EA hydrology station has a GEOGRAPHY point. We find the nearest Met Office
pseudo-observation site to each, enabling direct comparison of air temperature
(Met Office) vs water temperature (EA) at matched locations.


In [ ]:
-- For each Thames EA station, find the nearest Met Office site
SELECT
    s.LABEL AS ea_station,
    ps.SITE_NAME AS nearest_met_site,
    ROUND(ST_DISTANCE(s.LOCATION, ps.LOCATION), 0) AS distance_m
FROM MET_SCRATCH.THAMES.STATIONS s
CROSS JOIN MET_SCRATCH.THAMES.PSEUDO_SITES_GEO ps
WHERE s.RIVER_NAME ILIKE '%thames%'
QUALIFY ROW_NUMBER() OVER (PARTITION BY s.ID ORDER BY ST_DISTANCE(s.LOCATION, ps.LOCATION)) = 1
ORDER BY ea_station;


---
## Summary

| Layer | Source | Key Metric |
|-------|--------|-----------|
| River geometry | OS Open Rivers → `RIVER_LINKS` | 376 named Thames links, ~350km |
| Water observations | EA Hydrology API → `READINGS` | 3.3M readings across 34 stations |
| Air climate signal | Met Office pseudo-obs → `THAMES_TEMPERATURE_TIMESERIES` | 15.6M hourly obs, 116 sites, 2011–2026 |
| Ecological thresholds | Literature → `FISHDB.THAMES_10` | 10 species with stress/lethal temp bands |
| Population impact | Simulation → `THAMES_FISH_POPULATION_SUMMARY` | Baseline vs +3°C vs +5°C scenarios |

The spatial join (Part 5) is the bridge between the EA water monitoring network and the Met Office
climate grid — enabling calibration of the air-to-water temperature relationship that underpins
the fish thermal risk model.
